# 字符串的基础检索与转换

In [1]:
import os; nb_dir = os.getcwd(); 
import sys; 
sys.path.append(
    os.sep.join([nb_dir, os.pardir, os.pardir, "Customized_Package"])
); 

In [2]:
import ipynb_quick_render

In [3]:
motto = "Verba volant, scripta manent"; 

## 用索引访问子串

### 用索引访问单个字符

In [4]:
print(motto)
print(motto[0], motto[3], motto[-1], motto[-6]); 

Verba volant, scripta manent
V b t m


### 用索引蔟按照一定顺序访问元素, 使得访问的结果构成子表
* `start: end: step`中的末项, 是`end`索引的元素的上一个元素, \
    因此索引簇索引的是"左闭右开区间". 

In [5]:
print(motto)
print(motto[6: 11], motto[0: 5: 1], motto[-6: -1]); 
print(motto[-6: ], motto[: 6: 1]); 

Verba volant, scripta manent
volan Verba manen
manent Verba 


## 字符串中位置和计数信息的查询

* 查询字符串的长度 (包含的字符总数): `len(s)`或`s.__len__()`
    * 由于字符编码的差异, 通常情况下, 字符串的**字符数少于其字节数**; 
    * 对于**采用变长编码规则**的字符串, 一般无法直接通过将字符数乘以某个整数的方式推断其字节数. 
* 查询一个子串在字符串中...
    * ...是否存在: `chr in s`或`s.__contains__(chr)`
    * ...首次(或末次)出现的位序: `s.index(chr)`或`s.rindex(chr)`
    * ...出现的次数: `s.count(chr)`
    * ...是否位于其开头或者结尾: `s.startswith(chr)`或`s.endswith(chr)`

In [6]:
print(motto)
print(len(motto))
props = ("__contains__", "count", "index", "rindex", "startswith", "endswith"); 
vals = tuple(motto.__getattribute__(prop)("nt") for prop in props); 
ipynb_quick_render.Table2d(list(zip(props, vals))).render()

Verba volant, scripta manent
28


__contains__,True
count,2
index,10
rindex,26
startswith,False
endswith,True


## 字符串与元祖的异同
字符串与元组均为不可变对象, 因此无法对字符串中的个别字符或子串进行修改, 只能通过拼接, 分割, 替换等操作构建新的字符串. 

值得注意的是: 字符串也可作为可迭代对象用于构建`generator`, 每次**无间隔地顺次取出其中的一个字符**; 字符串还支持**与元组类似的批量挂载和变长挂载**语法. 在部分程序的挂载语句和循环语句中需要格外注意. 

字符串与元组的主要差异体现在: 元组可以形成多级嵌套结构, 但字符串中不同子串之间的地位是平等的. 

In [7]:
#字符串作为可迭代对象参与遍历
for char in "ABCD": 
    print(char * 2, end="\x20")

AA BB CC DD 

In [8]:
#字符串的批量挂载
a, b = "AB"; print(a, b); 
a, b, a = "ABC"; print(a); 

A B
C


In [9]:
#字符串的变长挂载
res = list(); 
for str_mnt in "AB", "ABC", "ABCD": 
    a, *b, c = str_mnt; 
    res.append((str_mnt, a, b, c)); 
ipynb_quick_render.Table2d(res).render(); 

AB,A,[],B
ABC,A,['B'],C
ABCD,A,"['B', 'C']",D


In [10]:
#在字符串中, 无论索引多少次, 所得的结果仍为字符串
sub_motto = motto[6:12]; subsub_motto = sub_motto[0]; 
print(subsub_motto == motto[6]); 
print(type(subsub_motto) is type(motto))

True
True


## 字符串的转换

### 字母的格式转换

此处所述的字母, 包括拉丁字母, 希腊字母, 西里尔字母, 格鲁吉亚字母等区分大小写的**字母**, 以及它们的声调形式和主要变体. 

以下格式转换函数仅适用于上述字母的大小写转换, 无法实现汉字的简体-繁体转换, 日文的平假名-片假名转换等功能. 

|函数|功能|备注|
|:-|:-:|:-|
|`s.upper()`|将字符串中的所有字母转换为大写||
|`s.lower()`|将字符串中的所有字母转换为小写||
|`s.title()`|将字符串中的所有单词中, <br>出现在词首的第一个字母转换为大写, <br>其余位置的字母转换为小写|单词是由**连续的欧洲拼音<br>文字字母**构成的子串; <br>不能识别单词的词性|

注意事项: 
* 部分连笔字母或者变调字母的小写形式中, 多个字母 (或字母与符号的组合) 被合成一个字符, 使用`upper`方法将该字符转化为大写形式后, 连写的字母 (或字母与符号的组合) 将被**拆分为对应的, 连续的多个大写字母** (或大写字母与符号的组合). 常见的案例包括:  
    * 小写字母`ß` (`U+00df`: 小写拉丁字母eszett) 是古字母`ſ` (`U+017f`: 小写拉丁字母长s) 和 `z` 的二合字母 (一说是`ſ`与`s`的二合字母)$^{[1–2]}$, 由于该字母不会用于德语单词的词首, 因此长期没有对应的大写字母, 其大写形式按照约定俗成, 采用其转写形式`ss`的大写, 即`SS`$^{[2]}$. 进入信息时代后, 在Unicode的发展过程中, 虽然新设计了该字母的大写形式`ẞ` (`U+1e9e`), 但出于兼容性方面的考虑, `ß` 的大写形式仍为`SS`, 而`ẞ`的小写形式为`ß`. 
    * 在部分印刷作品中, 小写字母组合`ff`, `fi`, `fl`, `ffi`, `ffl`分别连写成`ﬀ`, `ﬁ`, `ﬂ`, `ﬃ`, `ﬄ` (`U+fb00`~`U+fb04`), 仅使用一个字符$^{[1‚3–5]}$, 但对应的大写形式不连写, 占用多个字符. 
* 部分字母的大写 (或小写) 形式被**分配了多个码点**, 分别用于表示不同的含义, 它们的小写 (或大写) 形式**采用同一字符同一码点**. 常见的案例包括:  
    * 小写字母`μ`占据了两个码点 (`U+00b5`: 公制单位前缀"百万分之一", `U+03bc`: 小写希腊字母mu)$^{[6‚7]}$, 它们的大写形式均为`Μ` (`U+039c`: 大写希腊字母Mu); 而`Μ`使用`lower`方法所得的小写形式为`U+03bc`; 
    * 小写字母`σ` (`U+03c3`: 小写希腊字母sigma) 和小写字母`ς` (`U+03c2`: 专用于词尾的小写希腊字母sigma) 的大写形式均为`Σ`(`U+03a3`: 大写希腊字母Sigma)$^{[1‚8‚9]}$; 而`Σ`使用`lower`方法所得的小写形式为`U+03c3`; 
        * 与之类似的还有`θ`和`ϑ`, `φ`和`ϕ`, `π`和`ϖ` (后者专用于词首) , `ρ`和`ϱ` (后者专用于斜体) 等$^{[7]}$
        * 上述数对希腊字母中, 每对字母使用`upper`方法得到的大写字母, 均为同一字符同一码点, 但大写字母使用`lower`方法转化为小写形式时, 均采用字母对中的前一个字母
    * 大写字母`Å`占据了两个码点 (`U+00c5`: 大写拉丁字母A上方带圈, `U+212b`: 长度单位Ångstrom)$^{[9]}$, 它们的小写形式均为`å`(`U+00e5`: 小写拉丁字母a上方带圈); 而`å`使用`upper`方法所得的大写形式为`U+00c5`. 
        * 与之类似的还有表示电阻单位Ohm的`Ω`(`U+2126`)$^{[7]}$等其他兼用于计量单位符号的拉丁和希腊字母
    
在上述情况下, 大写字母和小写字母并非一一映射的关系, 因此`upper`方法和`lower`方法在遇到包含上述字母的字符串时, 不是互逆操作. 在进行部分语言的本地化适配, 或者处理部分科技类文章使用的符号时, 需要格外注意上述问题. 

参考文献: 
```
[1] Haralambous Y, Plaice J, Braams J. Never again active characters! 
    Ω-Babel[J]. TUGboat, 1995, 16(4): 418–427.

[2] Pitchford J. Dutch, German, Austrian, Flemish and Afrikaans names[J]. 
    The Indexer: The International Journal of Indexing, 2006, 25(2): C11–C14.

[3] Lubran A. Word Containing the FFL Trigram[J]. Word Ways, 1991, 24(2): 16.

[4] Hoenig A. Virtual fonts, virtuous fonts[J]. TUGboat, 1997, 18(2): 113–121.

[5] Rahma AMS, Bhaya WS, Al-Nasrawi DA. Data hiding method for english scripts 
    using ligature characters unicode[J]. 
    European Journal of Scientific Research, 2013, 112(4): 452–459.

[6] Lu CJ, Browne AC. Converting unicode lexicon and lexical tools for ASCII 
    NLP applications[C]//AMIA Annu Symp Proc. 2011, 2011: 1870.

[7] Foster MP. Principles for constructing notation in unit systems and 
    their application to the SI[J]. Accreditation and Quality Assurance, 
    2012, 17(1): 85–92.

[8] Haralambous Y. From Unicode to typography, a case study: 
    the Greek script[C]//Fourteenth International Unicode Conference. 
    1998: b–10.

[9] Moran S, Cysouw M. The Unicode Cookbook for Linguists: Managing writing 
    systems using orthography profiles[M]. Berlin: Language Science Press, 
    2017, 10.
```

In [11]:
#构造Unicode从U+0000到U+FFFF的所有字符顺次连接组成的字符串
char = str().join( 
    chr(x) for x in range(65535)
); 

In [12]:
#区分大小写形式的字符
str_case_diff = str().join(
    [ch for (x, ch) in enumerate(char) if ch.isupper() or ch.islower()]
); 
str_case_diff

'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyzªµºÀÁÂÃÄÅÆÇÈÉÊËÌÍÎÏÐÑÒÓÔÕÖØÙÚÛÜÝÞßàáâãäåæçèéêëìíîïðñòóôõöøùúûüýþÿĀāĂăĄąĆćĈĉĊċČčĎďĐđĒēĔĕĖėĘęĚěĜĝĞğĠġĢģĤĥĦħĨĩĪīĬĭĮįİıĲĳĴĵĶķĸĹĺĻļĽľĿŀŁłŃńŅņŇňŉŊŋŌōŎŏŐőŒœŔŕŖŗŘřŚśŜŝŞşŠšŢţŤťŦŧŨũŪūŬŭŮůŰűŲųŴŵŶŷŸŹźŻżŽžſƀƁƂƃƄƅƆƇƈƉƊƋƌƍƎƏƐƑƒƓƔƕƖƗƘƙƚƛƜƝƞƟƠơƢƣƤƥƦƧƨƩƪƫƬƭƮƯưƱƲƳƴƵƶƷƸƹƺƼƽƾƿǄǆǇǉǊǌǍǎǏǐǑǒǓǔǕǖǗǘǙǚǛǜǝǞǟǠǡǢǣǤǥǦǧǨǩǪǫǬǭǮǯǰǱǳǴǵǶǷǸǹǺǻǼǽǾǿȀȁȂȃȄȅȆȇȈȉȊȋȌȍȎȏȐȑȒȓȔȕȖȗȘșȚțȜȝȞȟȠȡȢȣȤȥȦȧȨȩȪȫȬȭȮȯȰȱȲȳȴȵȶȷȸȹȺȻȼȽȾȿɀɁɂɃɄɅɆɇɈɉɊɋɌɍɎɏɐɑɒɓɔɕɖɗɘəɚɛɜɝɞɟɠɡɢɣɤɥɦɧɨɩɪɫɬɭɮɯɰɱɲɳɴɵɶɷɸɹɺɻɼɽɾɿʀʁʂʃʄʅʆʇʈʉʊʋʌʍʎʏʐʑʒʓʕʖʗʘʙʚʛʜʝʞʟʠʡʢʣʤʥʦʧʨʩʪʫʬʭʮʯʰʱʲʳʴʵʶʷʸˀˁˠˡˢˣˤͅͰͱͲͳͶͷͺͻͼͽͿΆΈΉΊΌΎΏΐΑΒΓΔΕΖΗΘΙΚΛΜΝΞΟΠΡΣΤΥΦΧΨΩΪΫάέήίΰαβγδεζηθικλμνξοπρςστυφχψωϊϋόύώϏϐϑϒϓϔϕϖϗϘϙϚϛϜϝϞϟϠϡϢϣϤϥϦϧϨϩϪϫϬϭϮϯϰϱϲϳϴϵϷϸϹϺϻϼϽϾϿЀЁЂЃЄЅІЇЈЉЊЋЌЍЎЏАБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяѐёђѓєѕіїјљњћќѝўџѠѡѢѣѤѥѦѧѨѩѪѫѬѭѮѯѰѱѲѳѴѵѶѷѸѹѺѻѼѽѾѿҀҁҊҋҌҍҎҏҐґҒғҔҕҖҗҘҙҚқҜҝҞҟҠҡҢңҤҥҦҧҨҩҪҫҬҭҮүҰұҲҳҴҵҶҷҸҹҺһҼҽҾҿӀӁӂӃӄӅӆӇӈӉӊӋӌӍӎӏӐӑӒӓӔӕӖӗӘәӚӛӜӝӞӟӠӡӢӣӤӥӦӧӨөӪӫӬӭӮӯӰӱӲӳӴӵӶӷӸӹӺӻӼӽӾӿԀԁԂԃԄԅԆԇԈԉԊԋԌԍԎԏԐԑԒԓԔԕԖԗԘԙԚԛԜԝԞԟԠԡԢԣԤԥԦԧԨԩԪԫԬԭԮԯԱԲԳԴԵԶԷԸԹԺԻԼԽԾԿՀՁՂՃ

In [13]:
#大小写转换不互逆的小写字母
abnormal_lower = list(
    (hex(x), ch, ch.upper()) for (x, ch) in enumerate(char) 
    if ch.islower() and ch != ch.upper().lower()
); 
ipynb_quick_render.Table2d(abnormal_lower, multi_column=5).render(); 

0xb5,µ,Μ,0x1c82,ᲂ,О,0x1f83,ᾃ,ἋΙ,0x1fa7,ᾧ,ὯΙ,0x1fe7,ῧ,Ϋ͂
0xdf,ß,SS,0x1c83,ᲃ,С,0x1f84,ᾄ,ἌΙ,0x1fb2,ᾲ,ᾺΙ,0x1ff2,ῲ,ῺΙ
0x131,ı,I,0x1c84,ᲄ,Т,0x1f85,ᾅ,ἍΙ,0x1fb3,ᾳ,ΑΙ,0x1ff3,ῳ,ΩΙ
0x149,ŉ,ʼN,0x1c85,ᲅ,Т,0x1f86,ᾆ,ἎΙ,0x1fb4,ᾴ,ΆΙ,0x1ff4,ῴ,ΏΙ
0x17f,ſ,S,0x1c86,ᲆ,Ъ,0x1f87,ᾇ,ἏΙ,0x1fb6,ᾶ,Α͂,0x1ff6,ῶ,Ω͂
0x1f0,ǰ,J̌,0x1c87,ᲇ,Ѣ,0x1f90,ᾐ,ἨΙ,0x1fb7,ᾷ,Α͂Ι,0x1ff7,ῷ,Ω͂Ι
0x345,ͅ,Ι,0x1c88,ᲈ,Ꙋ,0x1f91,ᾑ,ἩΙ,0x1fbe,ι,Ι,0xfb00,ﬀ,FF
0x390,ΐ,Ϊ́,0x1e96,ẖ,H̱,0x1f92,ᾒ,ἪΙ,0x1fc2,ῂ,ῊΙ,0xfb01,ﬁ,FI
0x3b0,ΰ,Ϋ́,0x1e97,ẗ,T̈,0x1f93,ᾓ,ἫΙ,0x1fc3,ῃ,ΗΙ,0xfb02,ﬂ,FL
0x3c2,ς,Σ,0x1e98,ẘ,W̊,0x1f94,ᾔ,ἬΙ,0x1fc4,ῄ,ΉΙ,0xfb03,ﬃ,FFI
0x3d0,ϐ,Β,0x1e99,ẙ,Y̊,0x1f95,ᾕ,ἭΙ,0x1fc6,ῆ,Η͂,0xfb04,ﬄ,FFL


In [14]:
#大小写转换不互逆的大写字母
abnormal_upper = list(
    (hex(x), ch, ch.lower()) for (x, ch) in enumerate(char) 
    if ch.isupper() and ch != ch.lower().upper()
); 
ipynb_quick_render.Table2d(abnormal_upper).render(); 

0x130,İ,i̇
0x3f4,ϴ,θ
0x1e9e,ẞ,ß
0x2126,Ω,ω
0x212a,K,k
0x212b,Å,å
